
    Detection with yolov8
    
    Tracking with BoT-SORT 
    
    Counting using supervision
    

In [1]:
#!pip install ultralytics

import ultralytics
ultralytics.__version__

Creating new Ultralytics Settings v0.0.8 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\nandh\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


'8.4.173'

In [5]:
%pip install supervision

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import supervision
print("supervision.__version__:", supervision.__version__)

C:\Users\nandh\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


supervision.__version__: 0.30.7


In [3]:
import torch
torch.__version__

'2.0.1+cu117'

In [7]:
torch.cuda.get_device_name(0)

NameError: name 'torch' is not defined

In [8]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch version: 2.14.1+cpu
CUDA available: False


# Detection

In [9]:
from ultralytics import YOLO

# Load a pretrained YOLOv8n model
model = YOLO('yolov8n.pt')

# Run inference on 'bus.jpg' with arguments
model.predict(source="Testvideo.mp4", save=True, imgsz=320, conf=0.5)


WARNING 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/1803) D:\Personal Projects\Object detection\Testvideo.mp4: 192x320 6 cars, 356.4ms
video 1/1 (frame 2/1803) D:\Personal Projects\Object detection\Testvideo.mp4: 192x320 6 cars, 64.9ms
video 1/1 (frame 3/1803) D:\Personal Projects\Object detection\Testvideo.mp4: 192x320 7 cars, 65.6ms
video 1/1 (frame 4/1803) D:\Personal Projects\Object detection\Testvideo.mp4: 192x320 5 cars, 63.4ms
video 1/1 (frame 5/1803) D:\Personal Projects\Object dete

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 depth: None
 keypoints: None
 masks: None
 names: {0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle', 4: 'airplane', 5: 'bus', 6: 'train', 7: 'truck', 8: 'boat', 9: 'traffic light', 10: 'fire hydrant', 11: 'stop sign', 12: 'parking meter', 13: 'bench', 14: 'bird', 15: 'cat', 16: 'dog', 17: 'horse', 18: 'sheep', 19: 'cow', 20: 'elephant', 21: 'bear', 22: 'zebra', 23: 'giraffe', 24: 'backpack', 25: 'umbrella', 26: 'handbag', 27: 'tie', 28: 'suitcase', 29: 'frisbee', 30: 'skis', 31: 'snowboard', 32: 'sports ball', 33: 'kite', 34: 'baseball bat', 35: 'baseball glove', 36: 'skateboard', 37: 'surfboard', 38: 'tennis racket', 39: 'bottle', 40: 'wine glass', 41: 'cup', 42: 'fork', 43: 'knife', 44: 'spoon', 45: 'bowl', 46: 'banana', 47: 'apple', 48: 'sandwich', 49: 'orange', 50: 'broccoli', 51: 'carrot', 52: 'hot dog', 53: 'pizza', 54: 'donut', 55: 'cake', 56: 'chair', 57: 'couch', 

# Tracking 

In [16]:
from ultralytics import YOLO

# Configure the tracking parameters and run the tracker
model = YOLO('yolov8n.pt')

results = model.track(source="testing/d.mp4",conf=0.3, iou=0.5, save=True, tracker="bytetrack.yaml")  

# Tracking and counting  - 1 line

In [14]:
import cv2
from collections import defaultdict
import supervision as sv
from ultralytics import YOLO

# ============================================================
# 1. LOAD YOLO MODEL
# ============================================================

model = YOLO("yolov8n.pt")


# ============================================================
# 2. OPEN INPUT VIDEO
# ============================================================

video_path = "Testvideo.mp4"

cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print("Error: Could not open video.")
    exit()


# ============================================================
# 3. DEFINE COUNTING LINE
# ============================================================

# Adjust these coordinates if required.
# This line should cross the road.

START = sv.Point(350, 600)
END = sv.Point(1300, 600)


# ============================================================
# 4. TRACK HISTORY
# ============================================================

track_history = defaultdict(list)


# ============================================================
# 5. STORE VEHICLES THAT HAVE ALREADY BEEN COUNTED
# ============================================================

crossed_objects = set()


# ============================================================
# 6. VIDEO INFORMATION
# ============================================================

video_info = sv.VideoInfo.from_video_path(video_path)


# ============================================================
# 7. OUTPUT VIDEO
# ============================================================

with sv.VideoSink(
    "output_vehicle_count.mp4",
    video_info
) as sink:

    while cap.isOpened():

        success, frame = cap.read()

        if not success:
            break


        # ====================================================
        # 8. YOLO OBJECT TRACKING
        # ====================================================

        results = model.track(
            frame,
            classes=[2, 3, 5, 7],
            persist=True,
            tracker="bytetrack.yaml",
            verbose=False
        )


        # ====================================================
        # 9. DRAW YOLO DETECTIONS
        # ====================================================

        annotated_frame = results[0].plot()


        # ====================================================
        # 10. CHECK WHETHER TRACK IDS EXIST
        # ====================================================

        if results[0].boxes.id is not None:

            boxes = results[0].boxes.xywh.cpu()
            track_ids = results[0].boxes.id.int().cpu().tolist()

            class_ids = results[0].boxes.cls.int().cpu().tolist()


            # =================================================
            # 11. PROCESS EACH TRACKED VEHICLE
            # =================================================

            for box, track_id, class_id in zip(
                boxes,
                track_ids,
                class_ids
            ):

                x, y, w, h = box

                x = float(x)
                y = float(y)
                w = float(w)
                h = float(h)


                # =============================================
                # STORE TRACK HISTORY
                # =============================================

                track = track_history[track_id]

                track.append((x, y))


                # Keep only last 30 positions
                if len(track) > 30:
                    track.pop(0)


                # =============================================
                # CHECK VEHICLE CROSSING
                # =============================================

                if len(track) >= 2:

                    previous_x, previous_y = track[-2]

                    current_x, current_y = track[-1]


                    # Vehicle moving TOP → BOTTOM
                    if (
                        previous_y < START.y
                        and current_y >= START.y
                        and START.x <= current_x <= END.x
                    ):

                        # Count only once
                        if track_id not in crossed_objects:

                            crossed_objects.add(track_id)

                            print(
                                f"Vehicle crossed! "
                                f"ID: {track_id}"
                            )


                            # Highlight the vehicle
                            cv2.rectangle(
                                annotated_frame,

                                (
                                    int(x - w / 2),
                                    int(y - h / 2)
                                ),

                                (
                                    int(x + w / 2),
                                    int(y + h / 2)
                                ),

                                (0, 255, 0),

                                3
                            )


        # ====================================================
        # 12. DRAW COUNTING LINE
        # ====================================================

        cv2.line(
            annotated_frame,

            (START.x, START.y),
            (END.x, END.y),

            (0, 255, 0),

            3
        )


        # ====================================================
        # 13. DISPLAY COUNT
        # ====================================================

        count_text = (
            f"Vehicles crossed: "
            f"{len(crossed_objects)}"
        )


        cv2.putText(
            annotated_frame,

            count_text,

            (20, 50),

            cv2.FONT_HERSHEY_SIMPLEX,

            1.2,

            (0, 255, 0),

            3
        )


        # ====================================================
        # 14. WRITE FRAME TO OUTPUT VIDEO
        # ====================================================

        sink.write_frame(annotated_frame)


# ============================================================
# 15. RELEASE VIDEO
# ============================================================

cap.release()

print("====================================")
print("Processing completed!")
print("Output: output_vehicle_count.mp4")
print(f"Total vehicles crossed: {len(crossed_objects)}")
print("====================================")

Vehicle crossed! ID: 4
Vehicle crossed! ID: 5
Vehicle crossed! ID: 6
Vehicle crossed! ID: 56
Vehicle crossed! ID: 1
Vehicle crossed! ID: 7
Vehicle crossed! ID: 9
Vehicle crossed! ID: 12
Vehicle crossed! ID: 313
Vehicle crossed! ID: 505
Vehicle crossed! ID: 129
Vehicle crossed! ID: 101
Vehicle crossed! ID: 166
Vehicle crossed! ID: 11
Vehicle crossed! ID: 52
Vehicle crossed! ID: 1067
Vehicle crossed! ID: 1164
Vehicle crossed! ID: 783
Vehicle crossed! ID: 310
Vehicle crossed! ID: 708
Vehicle crossed! ID: 937
Vehicle crossed! ID: 502
Vehicle crossed! ID: 1288
Vehicle crossed! ID: 1326
Vehicle crossed! ID: 1054
Vehicle crossed! ID: 315
Vehicle crossed! ID: 2023
Vehicle crossed! ID: 1869
Vehicle crossed! ID: 1666
Vehicle crossed! ID: 2237
Vehicle crossed! ID: 2144
Vehicle crossed! ID: 2138
Vehicle crossed! ID: 2134
Vehicle crossed! ID: 1885
Vehicle crossed! ID: 1629
Vehicle crossed! ID: 1884
Vehicle crossed! ID: 2410
Vehicle crossed! ID: 2646
Vehicle crossed! ID: 2628
Vehicle crossed! ID: 29